<style>
.jp-Notebook .jp-MarkdownCell,
.text_cell_render {
    font-size: 14px;
    line-height: 1.38;
}
.jp-CodeCell .cm-editor,
.jp-CodeCell pre,
.code_cell .CodeMirror {
    font-size: 13px !important;
    line-height: 1.30 !important;
}
</style>

# ECON 422: Macroeconomics & Machine Learning
## Regularization

This short notebook focuses on using three regularization methods:

- **Ridge:** shrink coefficients with an $\ell_2$ penalty;
- **Lasso:** shrink coefficients and allow exact zeros with an $\ell_1$ penalty;
- **Elastic Net:** combine the $\ell_1$ and $\ell_2$ penalties.

The main goal is to fit the methods, vary their tuning parameters, and select a
model using cross-validation. Each small gray box is one step; type it into the blank
code cell immediately below.

### Setup

In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import (
    Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV
)
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.float_format', lambda x: f'{x:0.4f}')


def fit_and_measure(estimator):
    model = Pipeline([
        ('scale', StandardScaler()),
        ('model', estimator)
    ])
    model.fit(X_train, y_train)

    fitted = model.named_steps['model']
    summary = {
        'test_MSE': mean_squared_error(
            y_test, model.predict(X_test)
        ),
        'L2_norm': np.linalg.norm(fitted.coef_),
        'nonzero_coefficients': np.count_nonzero(
            np.abs(fitted.coef_) > 1e-8
        )
    }
    return model, summary

### Data

Run the provided cell. The outcome is future output growth. The predictors are
correlated macroeconomic variables plus several irrelevant candidates, so
regularization has something useful to do.

In [2]:
rng = np.random.default_rng(422)
n = 160

activity = rng.normal(size=n)
financial = rng.normal(size=n)

X = pd.DataFrame({
    'inflation': 0.65 * activity + rng.normal(scale=0.7, size=n),
    'unemployment': -0.70 * activity + rng.normal(scale=0.7, size=n),
    'policy_rate': 0.45 * activity + 0.40 * financial
                   + rng.normal(scale=0.7, size=n),
    'industrial_production': 0.80 * activity
                             + rng.normal(scale=0.6, size=n),
    'stock_return': 0.75 * financial + rng.normal(scale=0.8, size=n),
    'oil_price_growth': rng.normal(size=n)
})

for j in range(6):
    X[f'candidate_{j+1:02d}'] = (
        0.25 * activity + 0.20 * financial + rng.normal(size=n)
    )

beta_true = pd.Series(0.0, index=X.columns)
beta_true.loc[[
    'inflation', 'unemployment',
    'policy_rate', 'industrial_production'
]] = [1.2, -1.0, 0.7, 0.9]

y = pd.Series(
    X.to_numpy() @ beta_true.to_numpy()
    + rng.normal(scale=1.5, size=n),
    name='future_output_growth'
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=422
)

print('Training observations:', len(X_train))
print('Test observations:', len(X_test))
print('Predictors:', X.shape[1])

Training observations: 112
Test observations: 48
Predictors: 12


## Guided Practice

Work through Parts A--D as one integrated regularization practice.

### Part A — Ridge

Ridge uses the $\ell_2$ penalty
$\lambda\sum_j\beta_j^2$. Increase $\lambda$ and observe how the coefficient
norm changes.

#### Step A1 — Fit ridge models

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">ridge_rows = []

for lam in [0.01, 0.1, 1, 10, 100]:
    model, result = fit_and_measure(Ridge(alpha=lam))
    result[&#x27;lambda&#x27;] = lam
    ridge_rows.append(result)</pre>

In [3]:
# Type Step A1 — Fit ridge models here.
ridge_rows = []
for lam in [0.01, 0.1, 1, 10, 100]:
    model, result = fit_and_measure(Ridge(alpha=lam))
    result['lambda'] = lam
    ridge_rows.append(result)

#### Step A2 — View ridge results

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">ridge_results = pd.DataFrame(ridge_rows)
ridge_results[[&#x27;lambda&#x27;, &#x27;test_MSE&#x27;, &#x27;L2_norm&#x27;]]</pre>

In [4]:
# Type Step A2 — View ridge results here.
ridge_results = pd.DataFrame(ridge_rows)
ridge_results[['lambda', 'test_MSE', 'L2_norm']]

,lambda,test_MSE,L2_norm
0,0.0100,2.6910,2.0137
1,0.1000,2.6908,2.0124
2,1.0000,2.6897,1.9988
3,10.0000,2.7080,1.8821
4,100.0000,3.6516,1.3335


### Part B — Lasso

Lasso uses the $\ell_1$ penalty
$\lambda\sum_j|\beta_j|$. Increase $\lambda$ and observe how many coefficients
remain nonzero.

#### Step B1 — Fit lasso models

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">lasso_rows = []

for lam in [0.001, 0.01, 0.05, 0.1, 0.3]:
    model, result = fit_and_measure(
        Lasso(alpha=lam, max_iter=30000)
    )
    result[&#x27;lambda&#x27;] = lam
    lasso_rows.append(result)</pre>

In [5]:
# Type Step B1 — Fit lasso models here.
lasso_rows = []

for lam in [0.01, 0.1, 1, 10, 100]:
    model, result = fit_and_measure(Lasso(alpha=lam, max_iter = 30000))
    result['lambda'] = lam
    lasso_rows.append(result)

#### Step B2 — View lasso results

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">lasso_results = pd.DataFrame(lasso_rows)
lasso_results[
    [&#x27;lambda&#x27;, &#x27;test_MSE&#x27;, &#x27;nonzero_coefficients&#x27;]
]</pre>

In [6]:
# Type Step B2 — View lasso results here.
lasso_results = pd.DataFrame(lasso_rows)
lasso_results[['lambda', 'test_MSE', 'nonzero_coefficients']]

,lambda,test_MSE,nonzero_coefficients
0,0.0100,2.6541,12
1,0.1000,2.4571,7
2,1.0000,4.8567,4
3,10.0000,12.0043,0
4,100.0000,12.0043,0


### Part C — Elastic Net

Elastic net combines the $\ell_1$ and $\ell_2$ penalties. In scikit-learn,
`alpha` is $\lambda$ and `l1_ratio` controls the mixture. A larger `l1_ratio`
makes the model more lasso-like.

#### Step C1 — Fit elastic-net models

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">elastic_rows = []

for mix in [0.1, 0.5, 0.9]:
    model, result = fit_and_measure(
        ElasticNet(alpha=0.1, l1_ratio=mix, max_iter=30000)
    )
    result[&#x27;l1_ratio&#x27;] = mix
    elastic_rows.append(result)</pre>

In [7]:
# Type Step C1 — Fit elastic-net models here.
elastic_rows = []

for mix in [0.1, 0.5, 0.9]:
    model, result = fit_and_measure(
        ElasticNet(alpha=0.1, l1_ratio=mix, max_iter=30000)
    )
    result['l1_ratio'] = mix
    elastic_rows.append(result)

#### Step C2 — View elastic-net results

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">elastic_results = pd.DataFrame(elastic_rows)
elastic_results[
    [&#x27;l1_ratio&#x27;, &#x27;test_MSE&#x27;, &#x27;nonzero_coefficients&#x27;]
]</pre>

In [8]:
# Type Step C2 — View elastic-net results here.
elastic_results = pd.DataFrame(elastic_rows)
elastic_results[['l1_ratio', 'test_MSE', 'nonzero_coefficients']]

,l1_ratio,test_MSE,nonzero_coefficients
0,0.1000,2.6844,12
1,0.5000,2.5553,11
2,0.9000,2.4638,8


### Part D — Cross-validation

Use the same five folds for all three methods. Select $\lambda$ for ridge and
lasso, and select both $\lambda$ and `l1_ratio` for elastic net.

#### Step D1 — Create the folds

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">cv = KFold(n_splits=5, shuffle=True, random_state=422)</pre>

In [9]:
# Type Step D1 — Create the folds here.
cv = KFold(n_splits=5, shuffle=True, random_state=422)

#### Step D2 — Cross-validate ridge

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">ridge_cv, ridge_summary = fit_and_measure(
    RidgeCV(alphas=np.logspace(-3, 3, 100), cv=cv)
)
ridge_summary.update(
    model=&#x27;Ridge&#x27;,
    selected_lambda=ridge_cv.named_steps[&#x27;model&#x27;].alpha_,
    selected_l1_ratio=np.nan
)</pre>

In [11]:
# Type Step D2 — Cross-validate ridge here.
ridge_cv, ridge_summary = fit_and_measure(RidgeCV(alphas=np.logspace(-3,3,100), cv=cv))
ridge_summary.update(
    model='Ridge',
    selected_lambda=ridge_cv.named_steps['model'].alpha_,
    selected_l1_ratio=np.nan
)

#### Step D3 — Cross-validate lasso

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">lasso_cv, lasso_summary = fit_and_measure(
    LassoCV(
        alphas=np.logspace(-4, 0.5, 100),
        cv=cv, max_iter=30000
    )
)
lasso_summary.update(
    model=&#x27;Lasso&#x27;,
    selected_lambda=lasso_cv.named_steps[&#x27;model&#x27;].alpha_,
    selected_l1_ratio=np.nan
)</pre>

In [12]:
# Type Step D3 — Cross-validate lasso here.
lasso_cv, lasso_summary = fit_and_measure(LassoCV(alphas=np.logspace(-4,0.5,100), cv=cv, max_iter=30000))
lasso_summary.update(
    model='Lasso',
    selected_lambda=lasso_cv.named_steps['model'].alpha_,
    selected_l1_ratio=np.nan
)

#### Step D4 — Cross-validate elastic net

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">elastic_cv, elastic_summary = fit_and_measure(
    ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.9],
        alphas=np.logspace(-4, 0.5, 100),
        cv=cv, max_iter=30000
    )
)</pre>

In [14]:
# Type Step D4 — Cross-validate elastic net here.
elastic_cv, elastic_summary = fit_and_measure(ElasticNetCV( l1_ratio=[0.1, 0.5, 0.9], alphas = np.logspace(-4,0.5,100), cv=cv, max_iter=30000))

#### Step D5 — Compare selected models

<pre style="color:#8a8a8a;background:#f7f7f7;border:1px solid #e4e4e4;border-radius:5px;padding:9px 11px;margin:6px 0 8px 0;font-family:Menlo, Monaco, Consolas, 'Courier New', monospace;font-size:12.5px;line-height:1.32;white-space:pre;overflow-x:auto;">elastic_summary.update(
    model=&#x27;Elastic Net&#x27;,
    selected_lambda=elastic_cv.named_steps[&#x27;model&#x27;].alpha_,
    selected_l1_ratio=elastic_cv.named_steps[&#x27;model&#x27;].l1_ratio_
)

cv_results = pd.DataFrame([
    ridge_summary, lasso_summary, elastic_summary
])
cv_results.sort_values(&#x27;test_MSE&#x27;)</pre>

In [15]:
# Type Step D5 — Compare selected models here.
elastic_summary.update(
    model='ElasticNet',
    selected_lambda=elastic_cv.named_steps['model'].alpha_,
    selected_l1_ratio=elastic_cv.named_steps['model'].l1_ratio_
)

cv_results = pd.DataFrame([
    ridge_summary, lasso_summary, elastic_summary
])
cv_results.sort_values('test_MSE')

,test_MSE,L2_norm,nonzero_coefficients,model,selected_lambda,selected_l1_ratio
1,2.5182,1.8554,6,Lasso,0.1520,NaN
2,2.5372,1.8334,6,ElasticNet,0.1688,0.9000
0,2.7512,1.8040,12,Ridge,17.4753,NaN


### Quick summary

- Ridge usually keeps all predictors but shrinks their coefficients.
- Lasso can create a sparse model by setting coefficients to zero.
- Elastic net is a compromise that is useful when predictors are correlated.
- Standardize predictors before applying these penalties.
- Use cross-validation to choose the tuning parameters.

## Exercise

Without using a gray guide, compare ridge, lasso, and elastic net using
**four-fold cross-validation**. For elastic net, use
`l1_ratio=[0.25, 0.50, 0.75]`.

Complete the five blanks in the code cell below. Then identify the model with
the lowest test MSE and the model with the fewest nonzero coefficients.

In [21]:
# Packages
import numpy as np
import pandas as pd

from sklearn.linear_model import (
    Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV
)
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [25]:
# Copy and paste new data-generating-process from 'Quizzes: (Sep9) Exercise'
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(22)

n = 120
n_blocks = 8
block_size = 5
rho = 0.75

# Generate 40 predictors in correlated groups of five
X_blocks = []

for _ in range(n_blocks):
    common_factor = rng.normal(size=(n, 1))
    individual_noise = rng.normal(size=(n, block_size))

X_block = (
np.sqrt(rho) * common_factor
+ np.sqrt(1 - rho) * individual_noise
)
X_blocks.append(X_block)

X = np.hstack(X_blocks)

# Only 8 of the 40 predictors affect y
beta_true = np.zeros(n_blocks * block_size)
beta_true[:5] = [1.10, 0.95, 0.80, 0.65, 0.50]
beta_true[5:8] = [-1.00, -0.75, -0.50]

epsilon = rng.normal(loc=0, scale=1.3, size=n)
y = X @ beta_true + epsilon

# Give the data convenient labels
feature_names = [f"x{j:02d}" for j in range(1, X.shape[1] + 1)]
X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name="y")

X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.30,
random_state=7
)

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))
print("Number of predictors:", X.shape[1])

pd.DataFrame(
{"predictor": feature_names, "true_coefficient": beta_true}
).query("true_coefficient != 0")
y,
test_size=0.30,
random_state=7

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))
print("Number of predictors:", X.shape[1])

pd.DataFrame(
{"predictor": feature_names, "true_coefficient": beta_true}
).query("true_coefficient != 0")

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 40 is different from 5)

### Hints

1. The exercise asks for **four-fold** cross-validation.
2. Standardize the predictors before fitting the penalized regressions.
3. Use the pipeline method that learns from `X_train` and `y_train`.
4. Use the pipeline method that generates predictions for `X_test`.
5. The fitted estimator stores its coefficient vector in an attribute ending with an underscore.

After completing the code:

- The model with the smallest `test_MSE` has the best test-set prediction.
- The model with the smallest `nonzero_coefficients` gives the sparsest solution.
- The cutoff `1e-8` treats extremely small numerical values as zero.

In [ ]:
cv_ex = KFold(
    n_splits=____,
    shuffle=True,
    random_state=7
)

estimators_ex = {
    'Ridge': RidgeCV(
        alphas=np.logspace(-3, 3, 100), cv=cv_ex
    ),
    'Lasso': LassoCV(
        alphas=np.logspace(-4, 0.5, 100),
        cv=cv_ex, max_iter=30000
    ),
    'Elastic Net': ElasticNetCV(
        l1_ratio=[0.25, 0.50, 0.75],
        alphas=np.logspace(-4, 0.5, 100),
        cv=cv_ex, max_iter=30000
    )
}

exercise_rows = []

for name, estimator in estimators_ex.items():
    model = Pipeline([
        ('scale', ____________()),
        ('model', estimator)
    ])

    model.____(X_train, y_train)
    fitted = model.named_steps['model']

    exercise_rows.append({
        'model': name,
        'test_MSE': mean_squared_error(
            y_test, model._______(X_test)
        ),
        'nonzero_coefficients': np.count_nonzero(
            np.abs(fitted._____) > 1e-8
        )
    })

exercise_results = pd.DataFrame(exercise_rows)
exercise_results